## demonstrate that we have a project

In [ ]:
# !pip install --upgrade pip
# !pip install --upgrade astropy # this project requires Astropy 8.0.2

In [ ]:
import numpy as np
import pylab as plt
from astropy.timeseries import LombScargle
from astropy.timeseries.periodograms.lombscargle.implementations.utils import trig_sum

In [ ]:
rng = np.random.default_rng(17) # the most random number

In [ ]:
"""
# Code from astropy + Claude
This code
- (a) does all four astropy L-S normalizations and 
- (b) also returns the complex amplitude at each frequency

## bugs:
- untested!
"""

"""
*Prototype: recover complex Lomb-Scargle amplitudes from the quantities the
astropy 'fast' implementation already computes.*

Provenance
----------
`ls_fast_with_amps` below is a modified copy of `lombscargle_fast` from astropy
(BSD-3-Clause licensed, astropy v8.0.2):

  https://github.com/astropy/astropy/blob/v8.0.2/astropy/timeseries/periodograms/lombscargle/implementations/fast_impl.py

Everything is copied unchanged from that function, except where marked
"NEW" (the amplitude computation and the extra return value) or
"SIMPLIFIED" (options that were removed). It calls astropy's own
`trig_sum`, imported rather than copied:

  https://github.com/astropy/astropy/blob/v8.0.2/astropy/timeseries/periodograms/lombscargle/implementations/utils.py

Convention: model(t) = offset + Ac*cos(2 pi f t) + As*sin(2 pi f t)
            complex amplitude Z = Ac - 1j*As   (so model = Re[Z exp(+i 2 pi f t)])
"""

def ls_fast_with_amps(t, y, dy, f0, df, Nf, fit_mean=True, **trig_kw):
    """
    WARNING: Claude wrote this

    # Fast Lomb-Scargle power in all four normalizations, plus complex amplitudes.

    Modified copy of astropy's `lombscargle_fast`:
    https://github.com/astropy/astropy/blob/v8.0.2/astropy/timeseries/periodograms/lombscargle/implementations/fast_impl.py

    SIMPLIFIED relative to the original: there is no `normalization` argument
    (all are returned), center_data is always True, and use_fft=True /
    algorithm="lra" are fixed.

    ## Returns
    powers : dict of ndarray
        Keys "standard", "model", "log", "psd" (astropy's normalizations).
    Z : complex ndarray
        Complex amplitudes, in the units of y (independent of normalization).
    """
    # --- COPIED from astropy lombscargle_fast: input setup and data centering
    if dy is None:
        dy = 1
    t, y, dy = np.broadcast_arrays(t, y, dy)
    w = dy**-2.0
    w /= w.sum()
    y = y - np.dot(w, y)

    # --- COPIED (simplified): trig_sum arguments (original also accepts
    # use_fft, algorithm and trig_sum_kwds as function arguments)
    kw = dict(f0=f0, df=df, use_fft=True, N=Nf, algorithm="lra", **trig_kw)

    # --- COPIED from astropy: step 1, functions of the time-shift tau
    Sh, Ch = trig_sum(t, w * y, **kw)
    S2, C2 = trig_sum(t, w, freq_factor=2, **kw)
    if fit_mean:
        S, C = trig_sum(t, w, **kw)
        tan2 = (S2 - 2 * S * C) / (C2 - (C * C - S * S))
    else:
        tan2 = S2 / C2

    # --- COPIED from astropy: trig identities giving cos/sin of omega*tau
    # (variable `tan_2omega_tau` in the original is named `tan2` here)
    S2w = tan2 / np.sqrt(1 + tan2 * tan2)
    C2w = 1 / np.sqrt(1 + tan2 * tan2)
    Cw = np.sqrt(0.5) * np.sqrt(1 + C2w)
    Sw = np.sqrt(0.5) * np.sign(S2w) * np.sqrt(1 - C2w)

    # --- COPIED from astropy: step 2, the periodogram (Zechmeister & Kurster)
    YY = np.dot(w, y**2)
    YC = Ch * Cw + Sh * Sw
    YS = Sh * Cw - Ch * Sw
    CC = 0.5 * (1 + C2 * C2w + S2 * S2w)
    SS = 0.5 * (1 - C2 * C2w - S2 * S2w)
    if fit_mean:
        CC -= (C * Cw + S * Sw) ** 2
        SS -= (S * Cw - C * Sw) ** 2
    power = YC * YC / CC + YS * YS / SS   # COPIED: un-normalized power

    # --- COPIED (restructured) from astropy's normalization branch. The
    # original picks ONE normalization via an if/elif chain; here all four
    # are computed from the same un-normalized power and returned together.
    powers = {
        "standard": power / YY,
        "model": power / (YY - power),
        "log": -np.log(1 - power / YY),
        "psd": power * 0.5 * (dy**-2.0).sum(),
    }

    # --- NEW (not in astropy): amplitudes. In the tau-shifted frame the sin and
    # cos columns are orthogonal, so the LS coefficients are just YC/CC and
    # YS/SS. Cw, Sw are cos/sin(omega*tau) already computed above.
    a, b = YC / CC, YS / SS
    # rotate back to the un-shifted frame (cos(wt), sin(wt)):
    Ac = a * Cw - b * Sw
    As = a * Sw + b * Cw
    return powers, Ac - 1j * As

def claudes_test():
    # WARNING: Claude wrote this.
    # NEW test script (not from astropy). Compares against astropy's public API:
    # LombScargle.power() and LombScargle.model_parameters(), defined in
    # https://github.com/astropy/astropy/blob/v8.0.2/astropy/timeseries/periodograms/lombscargle/core.py
    rng = np.random.default_rng(42)
    N = 300
    t = np.sort(rng.uniform(0, 100, N))               # uneven sampling
    dy = rng.uniform(0.5, 1.5, N)                      # heteroskedastic errors
    f_true = 0.137
    y = 3.0 + 2.0 * np.cos(2 * np.pi * f_true * t + 0.7) + 1.1 * np.sin(2 * np.pi * 0.31 * t) \
        + rng.normal(0, dy)

    f0, df, Nf = 0.01, 0.002, 400
    freq = f0 + df * np.arange(Nf)

    for fit_mean in (True, False):
        print(f"\n===== fit_mean={fit_mean} =====")
        ls = LombScargle(t, y, dy, fit_mean=fit_mean, center_data=True)

        # ground truth amplitudes: astropy's own (per-frequency) LS fit
        Z_ref = np.array([
            (lambda th: th[-1] - 1j * th[-2])(ls.model_parameters(f, units=False))
            for f in freq
        ])

        powers, Z_f = ls_fast_with_amps(t, y, dy, f0, df, Nf, fit_mean=fit_mean)
        p_f = powers["standard"]
        for norm, p in powers.items():
            p_ref = ls.power(freq, method="fast", normalization=norm)
            print(f"fast '{norm}' matches astropy fast:".ljust(40),
                  np.allclose(p, np.asarray(p_ref), rtol=1e-6))
        print("fast amps vs exact amps: max |dZ| =", np.abs(Z_f - Z_ref).max(),
              "  (peak |Z| =", np.abs(Z_ref).max(), ")")

    i = np.argmax(p_f)
    print(f"\npeak freq {freq[i]:.4f} (true {f_true}); |Z|={abs(Z_f[i]):.3f} (true 2.0)")

In [ ]:
claudes_test()

In [ ]:
# make *extremely unrealistic* toy fake data
N = 128
times = np.sort(rng.uniform(size=N)) * 5. * 365.25 # 5 years in days
period = 1.3 * np.pi * 365.25 # 1.5 * pi years in days
omega = 2. * np.pi / period
phi_0 = 5.2331 # radians
scan_angles = rng.uniform(size=N) * 2. * np.pi # one circle in radians
a = 2.155 # arbitrary units
psi_0 = 1.1 # radians
b = 1.344 # arbitrary units; same as a
thetas = np.array([a * np.cos(psi_0), a * np.sin(psi_0)])[None, :] * np.cos(omega * times[:, None] - phi_0) \
       + np.array([-b * np.sin(psi_0), b * np.cos(psi_0)])[None, :] * np.sin(omega * times[:, None] - phi_0)
ivars = 1.0 + 1.5 * rng.uniform(size=N)
errs = 1. / np.sqrt(ivars)
noises = rng.normal(size=N) * errs
ys = np.sum(thetas * np.vstack([np.cos(scan_angles - psi_0), np.sin(scan_angles - psi_0)]).T, axis=1)

In [ ]:
# check the (noiseless) internals of the generative model
plt.figure(figsize=(4, 4))
plt.scatter(thetas[:, 0], thetas[:, 1], c=times)
plt.axis("equal")

In [ ]:
# check that the data look "okay"
plt.figure(figsize=(8, 4))
plt.errorbar(times, ys, marker="none", color="k", linestyle="none", yerr=errs)
plt.scatter(times, ys, c=scan_angles)

In [ ]:
# take the standard periodogram
f0, df = 1. / 3560., 1. / (5. * 3650.) # over-sampling
fs = np.arange(f0, 1. / 30., df)
Nf = len(fs)
powers = LombScargle(times, ys, errs, normalization="psd").power(fs)

In [ ]:
# nothing to see here
plt.figure(figsize=(8, 4))
plt.axvline(period, c="g")
plt.plot(1. / fs, powers, "k-")

In [ ]:
# now hack the LSP
# Note that what we are doing is: weighting the data by the cosines and sines of the scan angles.
# And: we are re-weighting the inverse variances appropriately.
# NOTE: Divides are unstable here; should modify LSP to take inverse variances.
print("running A")
powersA, Zs_A = ls_fast_with_amps(times, ys * np.cos(scan_angles),
                         1. / np.sqrt(ivars * np.cos(scan_angles) ** 2), f0, df, Nf, fit_mean=True)
print("running B")
powersB, Zs_B = ls_fast_with_amps(times, ys * np.sin(scan_angles),
                         1. / np.sqrt(ivars * np.sin(scan_angles) ** 2), f0, df, Nf, fit_mean=True)
print("running1")
powers0 = LombScargle(times, ys * np.cos(scan_angles),
             1. / np.sqrt(ivars * np.cos(scan_angles) ** 2), normalization="psd").power(fs)
print("running2")
powers1 = LombScargle(times, ys * np.sin(scan_angles),
             1. / np.sqrt(ivars * np.sin(scan_angles) ** 2), normalization="psd").power(fs)

In [ ]:
# we have a signal!
plt.figure(figsize=(8, 4))
plt.axvline(period, c="g")
plt.plot(1. / fs, powersA["psd"], "r-", lw=4, alpha=0.5, label="astropy cos")
plt.plot(1. / fs, powersB["psd"], "b-", lw=4, alpha=0.5, label="astropy sin")
plt.plot(1. / fs, powers0, "r-", lw=1, label="astropy cos")
plt.plot(1. / fs, powers1, "b-", lw=1, label="astropy sin")
plt.plot(1. / fs, powers0 + powers1, "k-", lw=1)